# Justificativas para limpeza de dados

Limpeza executada em src/data_cleaning.py.

In [1]:
import numpy as np
import pandas as pd
import src.data_cleaning as dc

df = dc.df

## 1. Valores inválidos e parsing de colunas

As colunas númericas apresentam o caractere \N e números decimais com vírgula (strings). Valores inválidos causam erros no parsing de colunas. Os nulos são discutidos na sessão 3.

In [2]:
def not_is_num(num):
    try:
        float(num)
        return False
    except:
        return True

invalids = []
for col_name in dc.num_cols:
    counts_idx = df[col_name].value_counts().index
    not_is_num_list = list(map(not_is_num, counts_idx))
    invalids.append(list(counts_idx[not_is_num_list]))

pd.Series(invalids, index=dc.num_cols)

ano                                                                   []
vagas_ps                                                            [\N]
inscritos_ps                                                        [\N]
aprovados_ps                                                        [\N]
ingressantes_ps                                                     [\N]
vagas_sisu                                                          [\N]
ingressantes_sisu                                                   [\N]
ingressantes_provar                                                 [\N]
ingressantes_outros                                                 [\N]
matriculados           [0,00, \N, 21,00, 60,00, 121,50, 86,00, 4,00, ...
evadidos                                                            [\N]
concluintes                                                         [\N]
rel_cand_vaga          [0,00, \N, 1,00, 0,24, 0,33, 0,54, 0,48, 0,98,...
dtype: object

**Decisão**: Substituir \N por valor nulo, vírgula por ponto e fazer parsing adequado das colunas.

In [3]:
dc.fix_invalid_data()

## 2. Manipulação de colunas

Há as seguintes regras esperadas entre colunas numéricas:

\begin{align*}
    \text{vagas\_ps} &< \text{ingressantes\_ps} \\
    \text{aprovados\_ps} &< \text{ingressantes\_ps} \\
    \text{inscritos\_ps} &< \text{aprovados\_ps} \\
    \text{vagas\_sisu} &< \text{ingressantes\_sisu}
\end{align*}

Entretanto, existem exceções para cada regra. A que é menos desobecida cobre 8% dos dados, o que é relevante tendo em vista a baixa granularidade e tamanho do dataset. Além disso, dentre as quatro modalidades de inscrição (ps, sisu, provar e outros), essas colunas dizem respeito apenas a ingressantes por duas delas.

Obs.: As exceções também podem incluir valores nulos, não necessariamente comparação numérica.

In [4]:
def pct_rule(col1, col2):
    quant = len(df[df[col1] < df[col2]])
    print(f"{f'{col1} < {col2}:':<32}{quant/len(df):.2f}")

pct_rule('vagas_ps', 'ingressantes_ps')
pct_rule('aprovados_ps', 'ingressantes_ps')
pct_rule('inscritos_ps', 'aprovados_ps')
pct_rule('vagas_sisu', 'ingressantes_sisu')

vagas_ps < ingressantes_ps:     0.27
aprovados_ps < ingressantes_ps: 0.08
inscritos_ps < aprovados_ps:    0.08
vagas_sisu < ingressantes_sisu: 0.09


**Decisão**: Deletar as colunas vagas_ps, inscritos_ps, aprovados_ps, rel_cand_vaga e vagas_sisu, manter as colunas de ingressantes e criar uma coluna da soma total de ingressantes.

In [5]:
dc.manipulates_cols()

## 3. Valores nulos

No ano de 2026 não há concluintes, então existe apenas valor nulo nessa coluna (ver data_loading.py). Todas as outras colunas numéricas que possuem valores nulos (66% delas, 11 linhas) apresentam valores nulos nas mesmas linhas, tornando-as instâncias de pouca informação. Não existem outros valores nulos na coluna concluintes em outros anos além daqueles já contabilizados pelas 11 linhas.

In [6]:
# Desconsidera concluintes
cols_with_na = ['ingressantes_ps', 'ingressantes_sisu', 'ingressantes_provar', 'ingressantes_outros',
                'matriculados', 'evadidos']

for i in range(1, len(cols_with_na)-1):
    bools = []
    idxs = lambda j: df[df[cols_with_na[j]].isna()].index
    bools.extend(idxs(0) == idxs(i))

print(df.isna().sum())

if all(bools):
    print('\nNulos das colunas consideradas correspondem às mesmas linhas')

df_aux = df.drop(index=idxs(0))
na_num = df_aux.loc[df_aux['ano'] != 2026, 'concluintes'].isna().sum()
print('Nulos de concluintes em outros anos não já contabilizados:', na_num)


curso                    0
setor                    0
campus                   0
grau                     0
turno                    0
ano                      0
ingressantes_ps         11
ingressantes_sisu       11
ingressantes_provar     11
ingressantes_outros     11
ingressantes_total       0
matriculados            11
evadidos                11
concluintes            208
dtype: int64

Nulos das colunas consideradas correspondem às mesmas linhas
Nulos de concluintes em outros anos não já contabilizados: 0


**Decisão**: Deletar as 11 linhas com valores nulos.

In [7]:
dc.drop_na()